# YOLO26 학습 파이프라인 (Colab)


## 클래스
- earthquake_building_level0
- earthquake_building_level2
- road_collapse_level0
- road_collapse_level2
- traffic_congestion_level0
- traffic_congestion_level2
- typhoon_tree_level0
- typhoon_tree_level2
- rock


## 폴더 구조
```
Makertone/
├─ data
│  ├─ dataset
├─ notebook/train.ipynb
└─ src/*.py
```


In [1]:
# 코랩 환경 구축

# Colab Google Drive 마운트
import torch
import sys
import os
import importlib

sys.modules['imp'] = importlib

from google.colab import drive
drive.mount('/content/drive')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

%load_ext autoreload
%autoreload 2

Mounted at /content/drive
Using device: cuda


In [ ]:
from pathlib import Path
import sys
import os

# PROJECT_ROOT를 설정합니다. (src 폴더가 위치한 곳)
PROJECT_ROOT = Path('/content/drive/MyDrive/ict2026')
# edge 패키지를 인식하기 위해 부모 디렉토리(drone)도 경로에 추가합니다.
PARENT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PARENT_ROOT) not in sys.path:
    sys.path.insert(0, str(PARENT_ROOT))

# 현재 작업 디렉토리를 PROJECT_ROOT로 변경하여 상대 경로가 올바르게 작동하도록 합니다.
os.chdir(PROJECT_ROOT)

print(f'PROJECT_ROOT: {PROJECT_ROOT.resolve()}')
print(f'PARENT_ROOT: {PARENT_ROOT.resolve()}')
print(f'Current Working Directory: {os.getcwd()}')


PROJECT_ROOT: /content/drive/MyDrive/ict2026
PARENT_ROOT: /content/drive/MyDrive
Current Working Directory: /content/drive/MyDrive/ict2026


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Colab에서 필요한 패키지 설치
from src.train_pipeline import install_training_dependencies

install_training_dependencies()
print('Dependencies installed.')


Dependencies installed.


In [ ]:
from pathlib import Path
from src.config import TrainConfig
import torch

# 학습 config
cfg = TrainConfig(
    project_root=PROJECT_ROOT,
    dataset_root=Path('/content/drive/MyDrive/ict2026/makertone.yolo26'),
    dataset_yaml_path=Path('/content/drive/MyDrive/ict2026/makertone.yolo26/data.yaml'),
    model_name='yolov8n.pt', # YOLOv8 모델로 변경
    image_size=512,
    epochs=100,
    batch_size=16,
    workers=4,
    device='0' if torch.cuda.is_available() else 'cpu',  # Colab GPU 사용 또는 CPU
    run_name='yolo8n_disaster_detector', # 실행 이름도 YOLOv8에 맞춰 변경
)

cfg

TrainConfig(project_root=PosixPath('/content/drive/MyDrive/ict2026'), dataset_root=PosixPath('/content/drive/MyDrive/ict2026/makertone.yolo26'), dataset_yaml_path=PosixPath('/content/drive/MyDrive/ict2026/makertone.yolo26/data.yaml'), class_names=['earthquake_building_level0', 'earthquake_building_level2', 'road_collapse_level0', 'road_collapse_level2', 'traffic_congestion_level0', 'traffic_congestion_level2', 'typhoon_tree_level0', 'typhoon_tree_level2', 'rock'], model_name='yolov8n.pt', image_size=512, epochs=100, batch_size=16, workers=4, device='0', project_dir=PosixPath('runs/detect'), run_name='yolo8n_disaster_detector', patience=20, seed=42, exist_ok=True, pretrained=True, amp=True)

In [ ]:
# (선택) Google Drive -> Colab 로컬 디스크로 데이터셋 사전 복사
import shutil
import time
from pathlib import Path

USE_LOCAL_DATASET_STAGE = False
LOCAL_DATASET_ROOT = Path('/content/drive/MyDrive/ict2026/makertone.yolo26')

if USE_LOCAL_DATASET_STAGE and Path('/content').exists():
    source_dataset_root = (cfg.project_root / cfg.dataset_root).resolve()
    print(f'source dataset: {source_dataset_root}')
    print(f'local dataset : {LOCAL_DATASET_ROOT}')
    start = time.time()

    if LOCAL_DATASET_ROOT.exists():
        print('Local dataset already exists. Skip copy.')
    else:
        LOCAL_DATASET_ROOT.parent.mkdir(parents=True, exist_ok=True)
        print('Copying dataset to local disk...')
        shutil.copytree(source_dataset_root, LOCAL_DATASET_ROOT)
        print(f'Copy done in {time.time() - start:.1f}s')

    cfg.dataset_root = LOCAL_DATASET_ROOT
    cfg.dataset_yaml_path = LOCAL_DATASET_ROOT / 'disaster.yaml'
    print(f'Updated cfg.dataset_root -> {cfg.dataset_root}')
else:
    print('Local dataset staging skipped.')


Local dataset staging skipped.


In [ ]:
import osㅇ

dataset_root_path = (cfg.project_root / cfg.dataset_root).resolve()
print(f"Checking dataset root: {dataset_root_path}")

if dataset_root_path.exists():
    print("\nContents of dataset root:")
    for item in os.listdir(dataset_root_path):
        print(f" - {item}")
else:
    print("\nDataset root DOES NOT EXIST!")


Checking dataset root: /content/drive/MyDrive/ict2026/makertone.yolo26

Contents of dataset root:
 - README.roboflow.txt
 - data.yaml
 - test
 - train
 - valid


In [ ]:
from src.train_pipeline import prepare_dataset_yaml

# Dataset 준비

dataset_yaml = prepare_dataset_yaml(cfg)
print(f'dataset yaml: {dataset_yaml}')
print('--- dataset yaml preview ---')
print(dataset_yaml.read_text(encoding='utf-8'))


dataset yaml: /content/drive/MyDrive/ict2026/makertone.yolo26/data.yaml
--- dataset yaml preview ---
path: /content/drive/MyDrive/ict2026/makertone.yolo26
train: train/images
val: valid/images
test: test/images

nc: 9
names: ['earthquake_building_level0', 'earthquake_building_level2', 'road_collapse_level0', 'road_collapse_level2', 'traffic_congestion_level0', 'traffic_congestion_level2', 'typhoon_tree_level0', 'typhoon_tree_level2', 'rock']



In [ ]:
from datetime import datetime
from pathlib import Path
from ultralytics import YOLO
import torch # Import torch to check cuda availability

# 모델 학습 + 에폭별 로그 출력
# Use cfg.dataset_yaml_path directly, which was confirmed correct in cell 3abf8826
final_dataset_yaml_path = cfg.dataset_yaml_path
project_dir = (cfg.project_root / cfg.project_dir).resolve()

# Determine the actual device to use
training_device = '0' if torch.cuda.is_available() else 'cpu'

print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Training start")
print(f"Using dataset yaml: {final_dataset_yaml_path}")
print(f"run name: {cfg.run_name}")
print(f"epochs={cfg.epochs}, batch={cfg.batch_size}, imgsz={cfg.image_size}, device={training_device}") # Log the actual device

model = YOLO(cfg.model_name)

def on_train_epoch_end(trainer):
    epoch = trainer.epoch + 1
    total_epochs = getattr(trainer, 'epochs', cfg.epochs)
    loss_items = trainer.label_loss_items(trainer.tloss, prefix='train')
    loss_log = ', '.join(f"{k}={float(v):.4f}" for k, v in loss_items.items())
    lr_log = ', '.join(f"{k}={float(v):.6f}" for k, v in trainer.lr.items())
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Epoch {epoch}/{total_epochs} | {loss_log} | {lr_log}")

def on_fit_epoch_end(trainer):
    epoch = trainer.epoch + 1
    total_epochs = getattr(trainer, 'epochs', cfg.epochs)
    metric_items = {
        k: v for k, v in trainer.metrics.items() if isinstance(v, (int, float))
    }
    if metric_items:
        metric_log = ', '.join(f"{k}={float(v):.4f}" for k, v in metric_items.items())
        print(f"[{datetime.now().strftime('%H:%M:%S')}] Val {epoch}/{total_epochs} | {metric_log}")

model.add_callback('on_train_epoch_end', on_train_epoch_end)
model.add_callback('on_fit_epoch_end', on_fit_epoch_end)

train_results = model.train(
    data=str(final_dataset_yaml_path), # Use the correct path from cfg
    imgsz=cfg.image_size,
    epochs=cfg.epochs,
    batch=cfg.batch_size,
    workers=cfg.workers,
    device=training_device, # Use the dynamically determined device
    project=str(project_dir),
    name=cfg.run_name,
    patience=cfg.patience,
    seed=cfg.seed,
    exist_ok=cfg.exist_ok,
    pretrained=cfg.pretrained,
    amp=cfg.amp,
    cache='disk',
    verbose=True,
)

run_dir = Path(train_results.save_dir) if hasattr(train_results, 'save_dir') else (project_dir / cfg.run_name)
best_weights = run_dir / 'weights' / 'best.pt'
print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Training finished")
print(f"run dir: {run_dir}")
print(f"best weights: {best_weights}")

[2026-05-09 18:11:04] Training start
Using dataset yaml: /content/drive/MyDrive/ict2026/makertone.yolo26/data.yaml
run name: yolo8n_disaster_detector
epochs=100, batch=16, imgsz=512, device=0
Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/ict2026/makertone.yolo26/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask

[autoreload of src.config failed: Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/extensions/autoreload.py", line 245, in check
    superreload(m, reload, self.old_objects)
  File "/usr/local/lib/python3.12/dist-packages/IPython/extensions/autoreload.py", line 394, in superreload
    module = reload(module)
             ^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/__init__.py", line 121, in reload
    raise ImportError(f"parent {parent_name!r} not in sys.modules",
ImportError: parent 'src' not in sys.modules
]


  7                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  8                  -1  1    460288  ultralytics.nn.modules.block.C2f             [256, 256, 1, True]           
  9                  -1  1    164608  ultralytics.nn.modules.block.SPPF            [256, 256, 5]                 
 10                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']          
 11             [-1, 6]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 12                  -1  1    148224  ultralytics.nn.modules.block.C2f             [384, 128, 1]                 
 13                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']          
 14             [-1, 4]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 15                  -1  1     37248  ultralytics.nn.modules.block.C2f             [192,

In [ ]:
# Raspberry Pi 배포를 위해 ONNX로 내보내기(선택)
from src.train_pipeline import export_model

onnx_path = export_model(best_weights, export_format='onnx')
print(f'onnx export: {onnx_path}')
